# 🧪 LAB: Machine Learning to Predict Concrete Compressive Strength

In this lab, you will apply `PyTorch` to a real-world data problem. You will implement a Neural Network in `PyTorch` using the new functionalities introduced this week and compare its performance with a Support Vector Machine and Linear Regression.

Your goal is to predict the **compressive strength of concrete** (the outcome variable, *Y*) based on a set of input features (*X*), including:
**Cement, Blast Furnace Slag, Fly Ash, Water, Superplasticizer, Coarse Aggregate, Fine Aggregate, and Age.**

This dataset and problem are based on the following paper:

- I-Cheng Yeh, "Modeling of strength of high performance concrete using artificial neural networks," Cement and Concrete Research, Vol. 28, No. 12, pp. 1797-1808 (1998). [Link to the paper](yeh1998.pdf).


---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.

NAME(s) = ""

In [16]:
%pip install xlrd

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 0. Overall Instructions 

In this lab, you will work with three models: a **Neural Network**, a **Support Vector Machine (SVM)**, and some form of **Linear Regression**. Each should meet the following requirements:

- **Do not tune the models**. For example, for the Neural Network, choose any number of layers and hidden units you prefer; for the SVM, select a kernel of your choice (e.g., linear, RBF, polynomial), etc.

- **Evaluate model performance using k-fold cross-validation**. That is, run each model using a chosen number of folds (*k*), and report the average performance across all folds. You may select the value of *k* that you want.

- **Be careful with data preprocessing**. Apply all preprocessing steps properly to avoid data leakage (e.g. standardizing features before data splitting)

## 1. Pre-implementation Group Discussion (15 points)

In your group, discuss, agree on, and elaborate the following points:

- **Descriptive analysis**. Identify what exploratory or descriptive analyses you can perform to better understand the relationship between each input variable and the outcome. Consider both visual (e.g., scatter plots, histograms) and statistical summaries.

- **Data preprocessing**. Decide what preprocessing steps are necessary given that you will be using neural networks. 

- **Model configuration**. Specify which cost function and output layer activation are most appropriate for your problem type (e.g., regression vs. classification). Explain your reasoning based on the nature of the outcome variable.

- **Performance evaluation**. Describe how you will perform k-fold cross-validation to assess your model’s performance. Include how you will divide the data, compute metrics across folds, and report the final averaged results.

USE AS MANY MARKDOWN CELLS AS NEEDED

## 2. Descriptive analysis (10 points)

Apply the descriptive analyses that your group agreed upon in the previous section.

**Load the dataset before beginning the descriptive analysis; you will also use it in the following sections.**

In [17]:
import numpy as np
import pandas as pd

# Load the data (8 input features, last column is the outcome)
df = pd.read_excel("Concrete_Data.xls")
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

## 3. Neural Network (40 points)

Implement a neural network as you did in the previous lab, but this time leverage the new functionalities introduced this week. Specifically:

- Your class implementing the neural network should inherit from nn.Module.

- Use `nn.Linear` for the linear transformations between layers, `nn.ReLU` for the hidden-layer activations, and the appropriate activation function for the output layer (e.g., `nn.Sigmoid` for binary classification).

- Use the appropriate cost function from `torch.nn` for your task (e.g., `nn.MSELoss`, `nn.BCELoss`).

- Use Adam from `torch.optim` as the optimizer.

- Use `DataLoader` to efficiently do batch processing. You may choose the batch size freely.

Once you have implemented this class, create an instance of it, then train and evaluate your model on the lab dataset.

**Remember to follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.**

In [18]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

class ConcreteNN(nn.Module):

    def __init__(self):
        super().__init__()
        l1 = nn.Linear(8, 32)
        a1 = nn.ReLU()
        l2 = nn.Linear(32, 16)
        a2 = nn.ReLU()
        l3 = nn.Linear(16, 1) # No activation for regression output
        l = [l1, a1, l2, a2, l3]
        self.module_list = nn.ModuleList(l)

    def forward(self, X):
        for f in self.module_list:
            X = f(X)
        return X

In [19]:
torch.manual_seed(1234)
kf = KFold(n_splits=5, shuffle=True, random_state=1234)
n_epochs = 200
batch_size = 32
rmse_list = []
r2_list = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    # Standardize inside each fold & fit on the training fold only to avoid data leakage
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y[train_idx], dtype=torch.float32).reshape(-1, 1)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = y[test_idx]

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = ConcreteNN()
    cost_function = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(n_epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_hat = model(x_batch)
            cost = cost_function(y_hat, y_batch)
            cost.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(X_test).numpy().flatten()

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    rmse_list.append(rmse)
    r2_list.append(r2)
    print(f"Fold {fold+1}: RMSE = {rmse:.3f}, R2 = {r2:.3f}")

print(f"Average RMSE = {np.mean(rmse_list):.3f}, Average R2 = {np.mean(r2_list):.3f}")

Fold 1: RMSE = 5.593, R2 = 0.876
Fold 2: RMSE = 5.361, R2 = 0.901
Fold 3: RMSE = 5.429, R2 = 0.899
Fold 4: RMSE = 6.293, R2 = 0.867
Fold 5: RMSE = 6.674, R2 = 0.827
Average RMSE = 5.870, Average R2 = 0.874


## 4. Replication (10 points)

Replicate your neural network experiment using the corresponding model class from `scikit-learn`: https://scikit-learn.org/stable/api/sklearn.neural_network.html. Be sure to apply the same neural network architecture as well as the same procedures for model evaluation and preprocessing to ensure a fair comparison.

**Remember**: Follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.

## 5. Comparison (20 points)

Apply a Support Vector Machine (SVM) and Linear Regression model to the dataset, and compare their performance with that of your Neural Network.

Discuss any similarities or differences you observe in their results and provide possible explanations for these patterns.

Be sure to apply the k-fold cross-validation procedure to ensure a fair comparison across all models.

**Remember**: Follow the general requirements regarding model tuning, evaluation, and data preprocessing described earlier.

In [20]:
import pandas as pd

df = pd.read_excel("Concrete_Data.xls")
df.columns

Index(['Cement (component 1)(kg in a m^3 mixture)',
       'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
       'Fly Ash (component 3)(kg in a m^3 mixture)',
       'Water  (component 4)(kg in a m^3 mixture)',
       'Superplasticizer (component 5)(kg in a m^3 mixture)',
       'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
       'Fine Aggregate (component 7)(kg in a m^3 mixture)', 'Age (day)',
       'Concrete compressive strength(MPa, megapascals) '],
      dtype='object')

In [21]:
# Support Vector Machine
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler


X = df[['Cement (component 1)(kg in a m^3 mixture)',
       'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
       'Fly Ash (component 3)(kg in a m^3 mixture)',
       'Water  (component 4)(kg in a m^3 mixture)',
       'Superplasticizer (component 5)(kg in a m^3 mixture)',
       'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
       'Fine Aggregate (component 7)(kg in a m^3 mixture)', 'Age (day)']]
y = df['Concrete compressive strength(MPa, megapascals) ']

svm_pipeline = make_pipeline(
    StandardScaler(),
    SVR(kernel="rbf")
)
cv = KFold(n_splits=5, shuffle=True, random_state=1234)

scores = cross_val_score(
    svm_pipeline,
    X,
    y,
    cv=cv,
    scoring="r2"
)

rmse_scores = cross_val_score(
    svm_pipeline,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

print("Average RMSE:", -rmse_scores.mean())
print("Average R²:", scores.mean())
print("Standard deviation:", scores.std())

Average RMSE: 9.912771214324325
Average R²: 0.6448456146594267
Standard deviation: 0.015611373085588031


In [22]:
# Linear
from sklearn.linear_model import LinearRegression

linear_pipeline = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

cv = KFold(n_splits=5, shuffle=True, random_state=1234)
scores = cross_val_score(
    linear_pipeline,
    X,
    y,
    cv=cv,
    scoring="r2"
)

rmse_scores = cross_val_score(
    linear_pipeline,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

print("Average RMSE:", -rmse_scores.mean())
print("Average R²:", scores.mean())
print("Standard deviation:", scores.std())

Average RMSE: 10.487398076438687
Average R²: 0.6020609424105023
Standard deviation: 0.024391086116698497


In [ ]:
# Ridge Regression (extra)
from sklearn.linear_model import Ridge
ridge_pipeline = make_pipeline(
    StandardScaler(),
    Ridge()
)

cv = KFold(n_splits=5, shuffle=True, random_state=1234)
scores = cross_val_score(
    ridge_pipeline,
    X,
    y,
    cv=cv,
    scoring="r2"
)

rmse_scores = cross_val_score(
    ridge_pipeline,
    X,
    y,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

print("Average RMSE:", -rmse_scores.mean())
print("Average R²:", scores.mean())
print("Standard deviation:", scores.std())

Average RMSE: 10.486830820003881
Average R²: 0.602131826815796
Standard deviation: 0.02406410074612057


## 6. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE